# 01. S2Vec 기초 실습

목표: S2Vec 논문의 핵심 전처리 흐름을 작은 합성 격자로 이해합니다.

실행 방법:

```bash
python -m pip install -r s2vec-geospatial-embeddings/requirements.txt
```

이 노트북은 실제 S2 Geometry나 Google Maps 데이터를 쓰지 않습니다. 16x16 격자를 S2 level 8 부모 셀 안의 level 12 자식 셀처럼 보고, 각 셀의 POI/도로 카운트를 합성합니다.

## 1. 작은 S2Vec 세계 만들기

논문에서는 level 8 셀 하나가 level 12 셀 16x16개를 포함합니다. 여기서는 그 구조만 흉내 냅니다. 각 셀은 `food`, `retail`, `office`, `transit`, `roads`, `parks`, `housing`, `industrial` 카운트를 가집니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

GRID = 16
FEATURES = ['food', 'retail', 'office', 'transit', 'roads', 'parks', 'housing', 'industrial']
F = len(FEATURES)

# 각 구역은 피처 카운트의 평균이 다릅니다.
# 실제 논문에서는 Google Maps 내부 피처를 세지만, 여기서는 학습용으로 포아송 분포에서 생성합니다.
ZONE_PROFILES = {
    'downtown': np.array([9, 8, 10, 7, 8, 1, 5, 1], dtype=float),
    'suburb': np.array([3, 4, 2, 2, 5, 4, 8, 1], dtype=float),
    'rural': np.array([1, 1, 0.5, 0.2, 2, 7, 2, 3], dtype=float),
    'park': np.array([1, 1, 0.2, 0.5, 1, 12, 0.5, 0.2], dtype=float),
}

def zone_for_cell(row, col):
    # 격자 중앙은 도심, 주변은 교외, 모서리는 농촌처럼 단순화합니다.
    center = np.array([(GRID - 1) / 2, (GRID - 1) / 2])
    dist = np.linalg.norm(np.array([row, col]) - center)
    if 5 <= row <= 8 and 10 <= col <= 13:
        return 'park'
    if dist < 3.2:
        return 'downtown'
    if dist < 6.2:
        return 'suburb'
    return 'rural'

feature_cube = np.zeros((GRID, GRID, F), dtype=float)
zones = np.empty((GRID, GRID), dtype=object)

for r in range(GRID):
    for c in range(GRID):
        zone = zone_for_cell(r, c)
        zones[r, c] = zone
        feature_cube[r, c] = rng.poisson(ZONE_PROFILES[zone])

print('feature cube shape:', feature_cube.shape)
print('one patch vector:', dict(zip(FEATURES, feature_cube[7, 7].astype(int))))

## 2. 피처 채널 시각화

S2Vec의 핵심은 지도 피처를 이미지처럼 다루는 것입니다. 아래 그림에서 각 채널은 한 종류의 피처 카운트를 나타냅니다.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)

for ax, name, idx in zip(axes.ravel(), FEATURES, range(F)):
    im = ax.imshow(feature_cube[:, :, idx], cmap='viridis')
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()

## 3. 래스터화와 정규화

논문에서는 각 patch cell의 피처 벡터를 row-by-row 순서로 배열합니다. 정규화는 카운트 범위가 큰 피처가 학습을 지배하지 않도록 하기 위한 기본 전처리입니다.

In [ ]:
patch_vectors = feature_cube.reshape(GRID * GRID, F)

# 피처별 평균과 표준편차로 정규화합니다.
# 실제 모델도 전역 통계로 피처를 정규화한 뒤 MAE를 학습합니다.
mean = patch_vectors.mean(axis=0)
std = patch_vectors.std(axis=0) + 1e-6
normalized_vectors = (patch_vectors - mean) / std
normalized_image = normalized_vectors.reshape(GRID, GRID, F)

print('patch vector matrix:', patch_vectors.shape)
print('rasterized image tensor:', normalized_image.shape)
print('row-major first 3 cells:')
for i in range(3):
    print(i, dict(zip(FEATURES, patch_vectors[i].astype(int))))

## 4. 마스킹 복원 직관

MAE는 일부 패치를 숨긴 뒤 주변 맥락으로 복원합니다. 아래 예시는 진짜 Transformer가 아니라, 주변 3x3 셀의 평균으로 가려진 셀을 채워 보는 기초 직관 실험입니다.

In [ ]:
mask_ratio = 0.50
mask = rng.random((GRID, GRID)) < mask_ratio

def neighbor_average_reconstruct(cube, mask):
    reconstructed = cube.astype(float).copy()
    global_mean = cube.reshape(-1, cube.shape[-1]).mean(axis=0)
    for r, c in np.argwhere(mask):
        r0, r1 = max(0, r - 1), min(GRID, r + 2)
        c0, c1 = max(0, c - 1), min(GRID, c + 2)
        local_mask = mask[r0:r1, c0:c1]
        local_values = cube[r0:r1, c0:c1, :][~local_mask]

        # 주변에 관측된 셀이 없으면 전체 평균을 사용합니다.
        # MAE의 encoder는 이보다 훨씬 강한 방식으로 넓은 문맥을 사용합니다.
        reconstructed[r, c] = local_values.mean(axis=0) if len(local_values) else global_mean
    return reconstructed

reconstructed = neighbor_average_reconstruct(feature_cube, mask)

channel = FEATURES.index('food')
masked_channel = feature_cube[:, :, channel].astype(float).copy()
masked_channel[mask] = np.nan

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for ax, title, data in [
    (axes[0], 'original food counts', feature_cube[:, :, channel]),
    (axes[1], 'masked input', masked_channel),
    (axes[2], 'simple reconstruction', reconstructed[:, :, channel]),
]:
    im = ax.imshow(data, cmap='magma')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()

In [ ]:
masked_original = feature_cube[mask]
masked_prediction = reconstructed[mask]
mae_by_feature = np.mean(np.abs(masked_prediction - masked_original), axis=0)

print('masked patch count:', int(mask.sum()))
print('mean absolute reconstruction error by feature')
for name, value in zip(FEATURES, mae_by_feature):
    print(f'{name:>10s}: {value:5.2f}')

## 정리

- S2Vec는 위치를 단일 좌표가 아니라 셀 안의 건조 환경 피처와 주변 공간 맥락으로 표현합니다.
- 16x16 패치 배열은 컴퓨터 비전 모델을 적용하기 위한 다리 역할을 합니다.
- 마스킹 복원은 라벨 없이도 주변 셀의 문법을 학습하게 만드는 자기지도 신호입니다.
- 다음 노트북에서는 이 문맥 피처가 다운스트림 예측에서 어떤 차이를 만드는지 실험합니다.